# IBM Data Science Capstone — Notebook 4: Exploratory Data Analysis with SQL

This notebook reproduces the **SQL analysis workflow** using the historical IBM `Spacex.csv` dataset (separate from the 90-row Falcon 9 machine-learning dataset). SQL queries execute on an in-memory SQLite database; results are displayed and exported for the final report.

**Source:** IBM Skills Network, `IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv`.

The figures and tables must be generated by running all cells. No results have been pre-filled.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
from IPython.display import display

DATA_URL = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
            "IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
OUTPUT = Path("spacex_sql_outputs")
OUTPUT.mkdir(exist_ok=True)

# Download the IBM course dataset; if unavailable, upload Spacex.csv manually.
try:
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from IBM Skills Network")
except Exception as e:
    print("Direct download unavailable:", e)
    print("Upload the official Spacex.csv file in the Colab dialog.")
    from google.colab import files
    uploaded = files.upload()
    match = next((k for k in uploaded if k.lower().endswith('.csv')), None)
    if match is None:
        raise ValueError("Please upload the official Spacex.csv")
    df = pd.read_csv(match)

# IBM dataset sometimes includes a Byte Order Mark / spaces.
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()
required = {"Launch_Site", "Orbit", "Mission_Outcome", "Landing_Outcome",
            "PAYLOAD_MASS__KG_", "Booster_Version"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Wrong dataset format; missing columns: {sorted(missing)}. Columns: {df.columns.tolist()}")
print("Dataset dimensions:", df.shape)
display(df.head())
display(df.isna().sum().rename("Null values").to_frame())


## SQLite setup

The table is named `SPACEXTBL`, as in the IBM assessment. Each query below is executable and its output is saved to CSV.

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

results = {}
def run_sql(name, sql):
    print(f"\n{name}\n{'-'*55}\n{sql.strip()}")
    table = pd.read_sql_query(sql, conn)
    display(table)
    table.to_csv(OUTPUT / f"{name}.csv", index=False)
    results[name] = table
    return table

run_sql("00_preview", "SELECT * FROM SPACEXTBL LIMIT 20;")


## Queries and evidence for the report

Keep each resulting table and add a short interpretation to the SQL slides. Counts reflect **this SQL dataset**, not necessarily the separate 90-launch classification dataset.

In [ ]:
run_sql("01_unique_launch_sites", """
SELECT DISTINCT Launch_Site
FROM SPACEXTBL
ORDER BY Launch_Site;
""")

run_sql("02_launches_per_site", """
SELECT Launch_Site, COUNT(*) AS Launch_Count
FROM SPACEXTBL
GROUP BY Launch_Site
ORDER BY Launch_Count DESC;
""")

run_sql("03_payload_min_max_avg", """
SELECT MIN(PAYLOAD_MASS__KG_) AS Minimum_Payload_kg,
       MAX(PAYLOAD_MASS__KG_) AS Maximum_Payload_kg,
       ROUND(AVG(PAYLOAD_MASS__KG_), 2) AS Average_Payload_kg
FROM SPACEXTBL;
""")

run_sql("04_total_payload_mass", """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass
FROM SPACEXTBL;
""")


In [ ]:
run_sql("05_mission_outcome_by_site", """
SELECT Launch_Site, Mission_Outcome, COUNT(*) AS Mission_Outcome_Count
FROM SPACEXTBL
GROUP BY Launch_Site, Mission_Outcome
ORDER BY Launch_Site, Mission_Outcome_Count DESC;
""")

run_sql("06_landing_outcomes", """
SELECT Landing_Outcome, COUNT(*) AS Landing_Count
FROM SPACEXTBL
GROUP BY Landing_Outcome
ORDER BY Landing_Count DESC;
""")

run_sql("07_successful_drone_ship_landings", """
SELECT COUNT(*) AS Successful_ASDS_Landings
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)';
""")

run_sql("08_orbit_counts", """
SELECT Orbit, COUNT(*) AS Launch_Count
FROM SPACEXTBL
GROUP BY Orbit
ORDER BY Launch_Count DESC;
""")

run_sql("09_geosynchronous_orbit", """
SELECT COUNT(*) AS GEO_Count
FROM SPACEXTBL
WHERE Orbit = 'GEO';
""")

run_sql("10_geostationary_transfer_orbit", """
SELECT COUNT(*) AS GTO_Count
FROM SPACEXTBL
WHERE Orbit = 'GTO';
""")


## Visualization from SQL results

The following chart is derived directly from the `COUNT(*) GROUP BY Launch_Site` query, not an illustrative mock-up.

In [ ]:
import matplotlib.pyplot as plt
site = results["02_launches_per_site"]
plt.figure(figsize=(10, 5))
plt.bar(site["Launch_Site"], site["Launch_Count"])
plt.xlabel("Launch site")
plt.ylabel("Launch count")
plt.title("SpaceX Launch Counts by Site — IBM SQL Dataset")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT / "sql_launches_by_site.png", dpi=180)
plt.show()


## Consistency checks and report notes

- `GEO` and `GTO` are different orbit labels. Do not interpret them as the same value.
- `Mission_Outcome` is **not** the same as `Landing_Outcome`.
- `Landing_Outcome = 'Success (drone ship)'` is specific to this IBM SQL dataset; the classification dataset uses labels such as `True ASDS`.
- The historical dataset might not reflect launches that occurred after the course snapshot.

In [ ]:
print("Rows in SQL source:", len(df))
print("Exported result tables:", len(results))
print("Files saved to:", OUTPUT.resolve())
print("SQL EDA notebook completed.")

# Download one archive containing all tables and chart, convenient for GitHub.
import shutil
archive = shutil.make_archive("spacex_sql_outputs", "zip", OUTPUT)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Archive:", archive)
